> **Development note:** This project includes explicit record counts, data previews, and reconciliation checks to make pipeline behavior observable during development and testing. In a production pipeline, unnecessary Spark actions such as repeated `count()` and `display()` calls would be removed or replaced with appropriate monitoring and data-quality metrics to reduce compute overhead.

# LA Parking Citations — Silver Transformation

This notebook implements the **Silver layer** of the LA Parking Citations geospatial data pipeline.

The pipeline is organized into three notebooks and orchestrated as a Databricks Lakeflow Job:

1. **01_ingest_bronze** — Ingest parking citation data from the LA Open Data API and persist the source data to a Bronze Delta table.
2. **02_transform_silver** — Clean, standardize, validate, and quarantine citation records to produce a Silver Delta table.
3. **03_enrich_gold** — Enrich valid citations with City Council district boundaries and create an analytics-ready Gold Delta table.

### Pipeline Architecture

LA Open Data API  
↓  
**Bronze** — Raw parking citation data  
↓  
**Silver** — Cleaned, typed, and validated citations  
↓  
**Gold** — Geospatially enriched citation data

## Notebook 2: Silver Transformation

This notebook reads from `parking_citations_bronze` and prepares citation records for downstream analysis and geospatial enrichment.

Silver processing includes:

- Converting source strings to appropriate data types
- Standardizing and cleaning source values
- Validating required fields and geographic coordinates
- Masking sensitive VIN data
- Separating invalid records into a quarantine table
- Persisting valid records to `parking_citations_silver`

**Input:** `parking_citations_bronze`  
**Outputs:** `parking_citations_silver` and the quarantine table

Geospatial enrichment and Gold-layer processing are handled by `03_enrich_gold`.

In [0]:
# View Bronze Table
from pyspark.sql import functions as F

bronze_df = spark.table("parking_citations_bronze")

print(f"Bronze records loaded: {bronze_df.count():,}")
display(bronze_df.limit(10))

Bronze records loaded: 50,000


ticket_number,issue_date,issue_time,meter_id,marked_time,rp_state_plate,plate_expiry_date,vin,make,body_style,color,location,route,agency,violation_code,violation_description,fine_amount,agency_desc,color_desc,body_style_desc,loc_lat,loc_long,geocodelocation
4603959006,2025-06-05T00:00:00.000,902,null,0000,CA,202409,null,DODG,VN,BK,20700 BRYANT ST,null,53,80.69BS,NO PARK/STREET CLEAN,73,53 - DOT - VALLEY,BLACK,VAN,34.2269981,-118.5861034,"{'type': 'Point', 'coordinates': [-118.5861034, 34.2269981]}"
4604675740,2025-06-05T00:00:00.000,837,null,0000,CA,202601,null,AUDI,PA,GY,3118 LA SALLE AVE S,null,55,80.69BS,NO PARK/STREET CLEAN,73,55 - DOT - SOUTHERN,GREY,PASSENGER CAR,34.02576288,-118.30555308,"{'type': 'Point', 'coordinates': [-118.30555308, 34.02576288]}"
4604443071,2025-06-05T00:00:00.000,845,null,0000,CA,202512,null,HYUN,PA,WT,3795 ROSEWOOD AV,null,51,80.69BS,NO PARK/STREET CLEAN,73,51 - DOT - WESTERN,WHITE,PASSENGER CAR,33.99900646,-118.44362349,"{'type': 'Point', 'coordinates': [-118.44362349, 33.99900646]}"
4604637715,2025-06-05T00:00:00.000,1233,null,0000,CA,202501,null,HYUN,PA,BK,1330 35TH PL W,null,55,80.69BS,NO PARK/STREET CLEAN,73,55 - DOT - SOUTHERN,BLACK,PASSENGER CAR,34.02371719,-118.29833515,"{'type': 'Point', 'coordinates': [-118.29833515, 34.02371719]}"
4604657960,2025-06-05T00:00:00.000,1040,null,0000,AZ,202509,null,TOYT,PA,BK,5216 ROSELAND ST,null,55,80.58L,PREFERENTIAL PARKING,68,55 - DOT - SOUTHERN,BLACK,PASSENGER CAR,34.02841185,-118.35784251,"{'type': 'Point', 'coordinates': [-118.35784251, 34.02841185]}"
4604096151,2025-06-05T00:00:00.000,856,HO667,0000,CA,202506,null,TOYT,PA,BL,1701 CHEROKEE AV,null,54,88.13B+,METER EXP.,63,54 - DOT - HOLLYWOOD,BLUE,PASSENGER CAR,34.1019943,-118.3347587,"{'type': 'Point', 'coordinates': [-118.3347587, 34.1019943]}"
4604217914,2025-06-03T00:00:00.000,1016,null,0000,CA,202510,null,TOYT,PA,WT,3120 71ST ST W,null,55,80.69BS,NO PARK/STREET CLEAN,73,55 - DOT - SOUTHERN,WHITE,PASSENGER CAR,33.9754104,-118.3273543,"{'type': 'Point', 'coordinates': [-118.3273543, 33.9754104]}"
4604735450,2025-06-05T00:00:00.000,1225,null,0000,CA,202405,null,HOND,PA,GY,1144 39TH ST W,null,55,80.69BS,NO PARK/STREET CLEAN,73,55 - DOT - SOUTHERN,GREY,PASSENGER CAR,34.0155554,-118.2956359,"{'type': 'Point', 'coordinates': [-118.2956359, 34.0155554]}"
4604527793,2025-06-05T00:00:00.000,856,null,0000,CA,202606,null,HOND,PA,SL,5000 8TH ST,null,54,80.69BS,NO PARK/STREET CLEAN,73,54 - DOT - HOLLYWOOD,SILVER,PASSENGER CAR,34.06051353,-118.33849479,"{'type': 'Point', 'coordinates': [-118.33849479, 34.06051353]}"
4604471966,2025-06-05T00:00:00.000,830,null,0000,CA,0,null,CHEV,PA,BK,1771 35TH ST W,null,55,80.69BS,NO PARK/STREET CLEAN,73,55 - DOT - SOUTHERN,BLACK,PASSENGER CAR,34.02478517,-118.31095343,"{'type': 'Point', 'coordinates': [-118.31095343, 34.02478517]}"


In [0]:
# TYPECASTING: Apply Silver Data Types
from pyspark.sql import functions as F

silver_df = (
    bronze_df
    .withColumn("issue_date", F.to_timestamp("issue_date"))
    .withColumn("issue_time", F.col("issue_time").cast("int"))
    .withColumn("agency", F.col("agency").cast("int"))
    .withColumn("fine_amount", F.col("fine_amount").cast("double"))
    .withColumn("loc_lat", F.col("loc_lat").cast("double"))
    .withColumn("loc_long", F.col("loc_long").cast("double"))
)

In [0]:
# CHECK
silver_df.printSchema()

root
 |-- ticket_number: string (nullable = true)
 |-- issue_date: timestamp (nullable = true)
 |-- issue_time: integer (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- marked_time: string (nullable = true)
 |-- rp_state_plate: string (nullable = true)
 |-- plate_expiry_date: string (nullable = true)
 |-- vin: string (nullable = true)
 |-- make: string (nullable = true)
 |-- body_style: string (nullable = true)
 |-- color: string (nullable = true)
 |-- location: string (nullable = true)
 |-- route: string (nullable = true)
 |-- agency: integer (nullable = true)
 |-- violation_code: string (nullable = true)
 |-- violation_description: string (nullable = true)
 |-- fine_amount: double (nullable = true)
 |-- agency_desc: string (nullable = true)
 |-- color_desc: string (nullable = true)
 |-- body_style_desc: string (nullable = true)
 |-- loc_lat: double (nullable = true)
 |-- loc_long: double (nullable = true)
 |-- geocodelocation: string (nullable = true)



In [0]:
# Normalize Issue Time
silver_df = silver_df.withColumn(
    "issue_time_formatted",
    F.concat(
        F.lpad((F.col("issue_time") / 100).cast("int"), 2, "0"),
        F.lit(":"),
        F.lpad((F.col("issue_time") % 100).cast("int"), 2, "0")
    )
)

In [0]:
# Check
display(
    silver_df.select(
        "ticket_number",
        "issue_time",
        "issue_time_formatted"
    ).limit(20)
)

ticket_number,issue_time,issue_time_formatted
4603959006,902,09:02
4604675740,837,08:37
4604443071,845,08:45
4604637715,1233,12:33
4604657960,1040,10:40
4604096151,856,08:56
4604217914,1016,10:16
4604735450,1225,12:25
4604527793,856,08:56
4604471966,830,08:30


In [0]:
# Validate Geographic Coordinates
silver_df = silver_df.withColumn(
    "valid_coordinates",
    (
        F.col("loc_lat").isNotNull()
        & F.col("loc_long").isNotNull()
        & F.col("loc_lat").between(33.0, 35.0)
        & F.col("loc_long").between(-119.0, -117.0)
    )
)

In [0]:
# Check
display(
    silver_df.groupBy("valid_coordinates")
    .count()
)

valid_coordinates,count
true,49954
false,46


In [0]:
# Validate other fields
silver_df = (
    silver_df
    .withColumn(
        "valid_ticket_number",
        F.col("ticket_number").isNotNull()
        & (F.trim(F.col("ticket_number")) != "")
    )
    .withColumn(
        "valid_issue_date",
        F.col("issue_date").isNotNull()
    )
    .withColumn(
        "valid_issue_time",
        F.col("issue_time").isNotNull()
        & ((F.col("issue_time") / 100).cast("int").between(0, 23))
        & ((F.col("issue_time") % 100).between(0, 59))
    )
    .withColumn(
        "valid_fine_amount",
        F.col("fine_amount").isNull()
        | (F.col("fine_amount") >= 0)
    )
    .withColumn(
        "valid_violation_code",
        F.col("violation_code").isNotNull()
        & (F.trim(F.col("violation_code")) != "")
    )
)

In [0]:
# Profile all fileds validated
validation_columns = [
    "valid_ticket_number",
    "valid_issue_date",
    "valid_issue_time",
    "valid_coordinates",
    "valid_fine_amount",
    "valid_violation_code"
]

display(
    silver_df.select([
        F.sum((~F.col(c)).cast("int")).alias(c)
        for c in validation_columns
    ])
)

valid_ticket_number,valid_issue_date,valid_issue_time,valid_coordinates,valid_fine_amount,valid_violation_code
0,0,0,46,0,1061


In [0]:
# Inspect invalid data
display(
    silver_df
    .filter(~F.col("valid_coordinates"))
    .select(
        "ticket_number",
        "issue_date",
        "location",
        "loc_lat",
        "loc_long",
        "geocodelocation"
    )
)

ticket_number,issue_date,location,loc_lat,loc_long,geocodelocation
2011324237,2025-05-27T00:00:00.000Z,NB SPRING ST @ TEMPLE ST,46.5653163,-66.4619164,"{'type': 'Point', 'coordinates': [-66.4619164, 46.5653163]}"
2011331277,2025-05-27T00:00:00.000Z,NB SPRING ST @ TEMPLE ST,46.5653163,-66.4619164,"{'type': 'Point', 'coordinates': [-66.4619164, 46.5653163]}"
2011327298,2025-05-27T00:00:00.000Z,NB SPRING ST @ TEMPLE ST,46.5653163,-66.4619164,"{'type': 'Point', 'coordinates': [-66.4619164, 46.5653163]}"
1115903585,2025-05-27T00:00:00.000Z,PASEO DEL MAR & WEYM,-33.0247597,-71.5547866,"{'type': 'Point', 'coordinates': [-71.5547866, -33.0247597]}"
2011332522,2025-05-27T00:00:00.000Z,NB SPRING ST @ TEMPLE ST,46.5653163,-66.4619164,"{'type': 'Point', 'coordinates': [-66.4619164, 46.5653163]}"
2011324691,2025-05-27T00:00:00.000Z,NB SPRING ST @ ARCADIA ST,46.5653163,-66.4619164,"{'type': 'Point', 'coordinates': [-66.4619164, 46.5653163]}"
1121166454,2024-10-18T00:00:00.000Z,4TH & VIRGIL ST,0.0,0.0,null
1132652253,2025-05-13T00:00:00.000Z,4TH PL & MOLINO,0.0,0.0,null
1131555412,2025-05-01T00:00:00.000Z,MLK & BUCKINGHAM RD,0.0,0.0,null
1132654552,2025-05-15T00:00:00.000Z,WINSTON & WALL ST,0.0,0.0,null


## Quarantine Invalid Records

Separate records that cannot safely proceed to geospatial enrichment.

Records with invalid critical fields are retained in quarantine rather than discarded so that source-data quality issues remain available for investigation.

In [0]:
silver_df = silver_df.withColumn(
    "quarantine_reason",
    F.when(~F.col("valid_ticket_number"), F.lit("INVALID_TICKET_NUMBER"))
     .when(~F.col("valid_issue_date"), F.lit("INVALID_ISSUE_DATE"))
     .when(~F.col("valid_issue_time"), F.lit("INVALID_ISSUE_TIME"))
     .when(~F.col("valid_coordinates"), F.lit("INVALID_COORDINATES"))
)

In [0]:
quarantine_df = silver_df.filter(
    F.col("quarantine_reason").isNotNull()
)

silver_valid_df = silver_df.filter(
    F.col("quarantine_reason").isNull()
)

In [0]:
# Verify we didn't lose any
total_count = silver_df.count()
valid_count = silver_valid_df.count()
quarantine_count = quarantine_df.count()

print(f"Total records:       {total_count:,}")
print(f"Valid records:       {valid_count:,}")
print(f"Quarantined records: {quarantine_count:,}")
print(f"Reconciled:          {valid_count + quarantine_count == total_count}")

Total records:       50,000
Valid records:       49,954
Quarantined records: 46
Reconciled:          True


In [0]:
# Mask VIN
from pyspark.sql import functions as F

silver_df = silver_df.withColumn(
    "vin_masked",
    F.when(
        F.col("vin").isNotNull(),
        F.concat(
            F.lit("*************"),
            F.substring("vin", -4, 4)
        )
    )
)

## Persist Silver and Quarantine Layers

Persist valid, cleaned citation records to the Silver Delta table and retain rejected records in a separate quarantine table for investigation and auditability.

In [0]:
# Drop
spark.sql("DROP TABLE IF EXISTS parking_citations_silver")
spark.sql("DROP TABLE IF EXISTS parking_citations_quarantine")

DataFrame[]

In [0]:
# Write Delta Tables
(
    silver_valid_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("parking_citations_silver")
)

(
    quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("parking_citations_quarantine")
)

In [0]:
# Reconcilliation Check
silver_table = spark.table("parking_citations_silver")
quarantine_table = spark.table("parking_citations_quarantine")

print(f"Silver records:     {silver_table.count():,}")
print(f"Quarantine records: {quarantine_table.count():,}")
print(
    f"Total persisted:    "
    f"{silver_table.count() + quarantine_table.count():,}"
)

display(quarantine_table)

Silver records:     49,954
Quarantine records: 46
Total persisted:    50,000


ticket_number,issue_date,issue_time,meter_id,marked_time,rp_state_plate,plate_expiry_date,vin,make,body_style,color,location,route,agency,violation_code,violation_description,fine_amount,agency_desc,color_desc,body_style_desc,loc_lat,loc_long,geocodelocation,issue_time_formatted,valid_coordinates,valid_ticket_number,valid_issue_date,valid_issue_time,valid_fine_amount,valid_violation_code,quarantine_reason
1132202503,2024-10-15T00:00:00.000Z,2305,null,0000,NV,202502,null,MERZ,PA,WH,S/W/C 69TH ST & FIGU,12A39,1,null,null,93.0,WESTERN,null,PASSENGER CAR,0.0,0.0,null,23:05,false,true,true,true,true,false,INVALID_COORDINATES
1132202514,2024-10-15T00:00:00.000Z,2310,null,0000,CA,202412,null,BMW,PA,BL,S/W/C 69TH ST & FIGU,12A39,1,null,null,93.0,WESTERN,BLUE,PASSENGER CAR,0.0,0.0,null,23:10,false,true,true,true,true,false,INVALID_COORDINATES
1132652253,2025-05-13T00:00:00.000Z,630,null,0000,CA,202405,null,null,PA,GY,4TH PL & MOLINO,null,1,4000A1,NO EVIDENCE OF REG,50.0,WESTERN,GREY,PASSENGER CAR,0.0,0.0,null,06:30,false,true,true,true,true,true,INVALID_COORDINATES
1131555412,2025-05-01T00:00:00.000Z,205,null,0000,CA,202407,null,MAZD,PA,BL,MLK & BUCKINGHAM RD,3A55,1,4000A1,NO EVIDENCE OF REG,50.0,WESTERN,BLUE,PASSENGER CAR,0.0,0.0,null,02:05,false,true,true,true,true,true,INVALID_COORDINATES
1132654552,2025-05-15T00:00:00.000Z,140,null,0000,CA,202511,null,CHEV,null,GY,WINSTON & WALL ST,null,1,8070,PK IN GRID LOCK ZN,163.0,WESTERN,GREY,null,0.0,0.0,null,01:40,false,true,true,true,true,true,INVALID_COORDINATES
2011288320,2025-05-22T00:00:00.000Z,1327,72000003,0000,CA,0,null,FRHT,null,bn,WB 6TH ST @ HARTFORD AVE,null,98,22500I,PK IN BUS ZONE,293.0,UNASSIGNED,null,null,22.9867569,87.8549755,"{'type': 'Point', 'coordinates': [87.8549755, 22.9867569]}",13:27,false,true,true,true,true,true,INVALID_COORDINATES
1129471582,2025-05-27T00:00:00.000Z,905,null,0000,CA,0,null,TSMR,PA,WH,CAYUGA & TUXFORD,null,1,8056E4,RED ZONE,93.0,WESTERN,null,PASSENGER CAR,42.9189555,-76.7263397,"{'type': 'Point', 'coordinates': [-76.7263397, 42.9189555]}",09:05,false,true,true,true,true,true,INVALID_COORDINATES
1131758821,2025-05-21T00:00:00.000Z,1845,null,0000,CA,202411,null,JEEP,PA,BU,G ST/BTWN ISLAND & F,HARB,1,4000A1,NO EVIDENCE OF REG,50.0,WESTERN,null,PASSENGER CAR,0.0,0.0,null,18:45,false,true,true,true,true,true,INVALID_COORDINATES
2011290755,2025-05-22T00:00:00.000Z,1258,72000003,0000,CA,0,null,FRHT,null,WT,WB 6TH ST @ HARTFORD AVE,null,98,22500I,PK IN BUS ZONE,293.0,UNASSIGNED,WHITE,null,22.9867569,87.8549755,"{'type': 'Point', 'coordinates': [87.8549755, 22.9867569]}",12:58,false,true,true,true,true,true,INVALID_COORDINATES
1132652264,2025-05-13T00:00:00.000Z,632,null,0000,CA,202501,null,TSMR,PA,GY,4TH PL & MATEO ST,null,1,4000A1,NO EVIDENCE OF REG,50.0,WESTERN,GREY,PASSENGER CAR,0.0,0.0,null,06:32,false,true,true,true,true,true,INVALID_COORDINATES
